In [33]:
"""
Importation necessaire:
WhisperProcessor: - Pour convertir l'audio en en spectrogrammes log-Mel.
                  - Pour transcire les tokens generes par le decodeur en texte.
WhisperForConditionalGeneration: - Pour telecharger le model whisper et l'instancier
load_dataset: - Pour charger un dataset, elle retourne un dict de deux cles, train contient les colonnes et num_arrows le nombre de lignes.
Audio: - Pour transformer un fichier audio en un tableau numpy
"""
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_dataset, Audio

ModuleNotFoundError: No module named 'evaluate'

### Remarque:
- Les datasets commonvoice utilise dans les scripts fourni par huggingface ne sont plus supporter par load_dataset  
  SOLUTION: On charge les donnees sur le Disk et on les recupere avec load_dataset et on fait un petit traitement pour utiliser
  le script sans le modifier.  
- Pour l'inference en anglais, Le script utilise un dataset qui ne pose pas ce probleme.  


In [23]:
student_id = !echo "$USER"

DATASETS_PATH = f"/info/raid-etu/m1/{student_id[0]}/projet-m1-asr/datasets/"
DATASETS_PATH

'/info/raid-etu/m1/s2100786/projet-m1-asr/datasets/'

In [16]:
import os
def load_and_process_data (DATASETS_PATH, FOLDER, FILE_NAME):
    # Charger le dataset
    dataset = load_dataset('csv', data_files=os.path.join("..", "datasets", FOLDER, f"{FILE_NAME}.tsv"), sep="\t")
    PATH_TO_AUDIO = DATASETS_PATH + FOLDER + "clips/"
    # Creer la colonne audio qui contient le chemin complet + le nom de fichier qui reside dans la colonne path
    dataset["train"] = dataset["train"].map( lambda x: {"audio": PATH_TO_AUDIO + x["path"]})
    """
    cast_columns: change le type d'une colonne
    Audio: permet de transformer un fichier audio en un tableau numpy avece un echantillonage de 16000 khz
    donc la colonne audio contiendra le tableau numpy qui sera apres transformer en spectogramme mel-log
    """
    dataset["train"] = dataset["train"].cast_column("audio", Audio(sampling_rate=16_000))
    return dataset

In [19]:
def whisper_inference (DATASETS_PATH, LANGUAGE, TASK, FOLDER, FILE_NAME):
    # load model and processor
    processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
    forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)

    # Recuperer le dataset
    ds = load_and_process_data(DATASETS_PATH, FOLDER, FILE_NAME)

    # Pour stocker les resultats
    results = []

    # Pour transcire les 10 premiers audio
    for i in range(10):  
        # recuperer la colonne audio qui contient le tableau numpy qui represente l'audio
        input_speech = ds["train"][i]["audio"]
        # transformer le tableau numpy en un spectogramme log-mel
        input_features = processor( input_speech["array"], sampling_rate=input_speech["sampling_rate"], return_tensors="pt").input_features
        # generer les tokens
        predicted_ids = model.generate(input_features, forced_decoder_ids=forced_decoder_ids)
        # decoder les tokens (transcire)
        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        # stocker les resultats
        results.append({
            "path": ds["train"][i]["path"], # chemin complet
            "sentence": ds["train"][i]["sentence"],  # transcription reelle (etiquette)
            "predicted": transcription  # transcription Whisper
        })
    # affichage
    for r in results:
        print(f"\n {r['path']}")
        print(f"Whisper: {r['predicted']}")
        print(f"label: {r['sentence']}")
        

### Inference En Francais

In [ ]:
LANGUAGE = "french"
TASK = "transcribe"
FOLDER = "francais/"
FILE_NAME = "validated"

whisper_inference (DATASETS_PATH, LANGUAGE, TASK, FOLDER, FILE_NAME)

### Inference En Arabe

In [11]:
LANGUAGE = "ar"
TASK = "transcribe"
FOLDER = "arabe/"
FILE_NAME = "validated"

whisper_inference (DATASETS_PATH, LANGUAGE, TASK, FOLDER, FILE_NAME)

Generating train split: 11778 examples [00:00, 143989.56 examples/s]
Map: 100%|██████████| 11778/11778 [00:01<00:00, 11613.46 examples/s]



 common_voice_ar_19216700.mp3
Whisper:  الادئيكة حظم
label: ألديك قلم ؟

 common_voice_ar_19242995.mp3
Whisper:  أين المشكل؟
label: أين المشكلة ؟

 common_voice_ar_21718692.mp3
Whisper:  نار بار
label: أربعة

 common_voice_ar_19235572.mp3
Whisper:  تاكمم بيه ذات بكي هكذا ميج لب
label: تقوم أمي بإعداد كعكة لأجل أبي.

 common_voice_ar_21818419.mp3
Whisper:  ستة
label: ستة

 common_voice_ar_20933525.mp3
Whisper:  ها هو العنوان
label: .ها هو العنوان

 common_voice_ar_21340998.mp3
Whisper:  كادت أن تتأخر عن المدرسة
label: كادت أن تتأخر عن المدرسة.

 common_voice_ar_19204574.mp3
Whisper:  هل يمكنني أن أصدقائي؟
label: كم الساعة ؟

 common_voice_ar_21352497.mp3
Whisper:  أينا الخسيل
label: أين الغسيل ؟

 common_voice_ar_20933539.mp3
Whisper:  أعرف تلك الفتاة
label: .أعرف تلك الفتاة


### Inference En Anglais

In [5]:
def english_whisper_inference():
    processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
    model.config.forced_decoder_ids = None
    ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
    sample = ds[0]["audio"]
    # `language="en"` pour choisir la transcription en anglais sans ambiguïté
    input_features = processor(sample["array"], sampling_rate=sample["sampling_rate"], return_tensors="pt", language="en").input_features 
    predicted_ids = model.generate(input_features)
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=False)
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
    print(transcription)


In [6]:
english_whisper_inference()

[' Mr. Quilter is the apostle of the middle classes, and we are glad to welcome his gospel.']
